# SORT1 rs12740374 — validation with CEBPA/CEBPB co-occupancy

Reproduces the MCP call ``analyze_variant_multilayer(...)``
for variant **chr1:109274968 G>T**
on the **alphagenome** oracle. Top-to-bottom execution
writes the same `example_output.md/json/tsv` and HTML report
the MCP tool would.

_Requires the `chorus` package installed and the `chorus-alphagenome` env
created (`chorus setup --oracle alphagenome`)._

In [ ]:
# All imports the notebook needs — top-level, no later imports.
import json
from pathlib import Path

import chorus
from chorus.analysis.normalization import get_normalizer
from chorus.analysis.variant_report import build_variant_report

In [ ]:
# Locate the walkthrough directory so artifacts land alongside the notebook.
WALKTHROUGH_DIR = Path.cwd()
print(f"Writing artifacts to: {WALKTHROUGH_DIR}")

In [ ]:
# Instantiate the oracle and load weights. ``use_environment=True``
# delegates the heavy model load to the matching mamba env
# (``chorus-alphagenome``) so the notebook itself runs in base chorus.
oracle = chorus.create_oracle(
    oracle_name='alphagenome',
    use_environment=True,
)
oracle.load_pretrained_model()

In [ ]:
# Define every input explicitly — no implicit defaults.
oracle_name = 'alphagenome'
position = 'chr1:109274968'
ref_allele = 'G'
alt_alleles = ['T']
gene_name = 'SORT1'
assay_ids = [
    'DNASE/EFO:0001187 DNase-seq/.',
    'ATAC/EFO:0001187 ATAC-seq/.',
    'CHIP_TF/EFO:0001187 TF ChIP-seq CEBPA genetically modified (insertion) using CRISPR targeting H. sapiens CEBPA/.',
    'CHIP_TF/EFO:0001187 TF ChIP-seq CEBPB/.',
    'CHIP_HISTONE/EFO:0001187 Histone ChIP-seq H3K27ac/.',
    'CAGE/hCAGE EFO:0001187/+',
    'CAGE/hCAGE EFO:0001187/-',
]

In [ ]:
# Auto-pick a prediction region of the oracle's natural width centred
# on the variant. Matches what the MCP tool does internally.
_chrom, _pos = position.split(":")
_pos = int(_pos)
_half = oracle.output_size // 2
region = f"{_chrom}:{max(1, _pos - _half)}-{_pos + _half}"

In [ ]:
# Run the oracle on ref + alt allele(s). Returns a dict with
# 'predictions', 'effect_sizes', and 'variant_info'.
variant_result = oracle.predict_variant_effect(
    genomic_region=region,
    variant_position=position,
    alleles=[ref_allele] + alt_alleles,
    assay_ids=assay_ids,
)

In [ ]:
# Build the multi-layer report — same function the MCP tool calls.
# get_normalizer() auto-loads the per-track CDF for percentile scoring.
normalizer = get_normalizer(oracle_name=oracle_name)
report = build_variant_report(
    variant_result=variant_result,
    oracle_name=oracle_name,
    gene_name=gene_name,
    normalizer=normalizer,
    igv_raw=False,
)

In [ ]:
# Save the same artifacts the MCP tool would produce:
#   - example_output.md  (markdown report)
#   - example_output.json (structured scores)
#   - example_output.tsv (track-level table)
#   - rs12740374_SORT1_CEBP_validation_report.html (interactive IGV report)
WALKTHROUGH_DIR.joinpath("example_output.md").write_text(report.to_markdown())
WALKTHROUGH_DIR.joinpath("example_output.json").write_text(
    json.dumps(report.to_dict(), indent=2, default=str)
)
try:
    report.to_dataframe().to_csv(
        WALKTHROUGH_DIR / "example_output.tsv", sep="\t", index=False,
    )
except Exception as exc:
    print(f"TSV write skipped: {exc}")

_html_path = report.to_html(output_path=str(WALKTHROUGH_DIR / "rs12740374_SORT1_CEBP_validation_report.html"))
print(f"Wrote artifacts to { WALKTHROUGH_DIR }")


## What this notebook produced

- `example_output.md` — markdown report (top tracks per layer)
- `example_output.json` — structured per-layer / per-track scores
- `example_output.tsv` — track-level table (one row per track)
- `rs12740374_SORT1_CEBP_validation_report.html` — interactive IGV browser with ref / alt overlays
